<a href="https://colab.research.google.com/github/zeeldhorajiya1909/codsoft_tasks/blob/main/codsoft_task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc
from sklearn.datasets import make_classification

# Install imbalanced-learn if you haven't already
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    print("Installing imbalanced-learn...")
    !pip install imbalanced-learn
    from imblearn.over_sampling import SMOTE

# Install kagglehub if you haven't already
try:
    import kagglehub
except ImportError:
    print("Installing kagglehub...")
    !pip install kagglehub
    import kagglehub

# --- 1. Load Data ---
# Using kagglehub to download the dataset
print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("kartik2112/fraud-detection")

print(f"Path to dataset files: {path}")

# List contents of the downloaded path to verify the CSV filename
import os
dataset_files = os.listdir(path)
print(f"Contents of {path}: {dataset_files}")

# Assuming the primary dataset for training is 'fraudTrain.csv'
# If you intend to use 'fraudTest.csv' or combine them, please let me know.
csv_file_path = os.path.join(path, 'fraudTrain.csv') # Corrected filename
df = pd.read_csv(csv_file_path)

print("\nDataset Head:")
print(df.head())
print("\nClass Distribution:")
print(df['is_fraud'].value_counts()) # Corrected column name
print("\nClass Distribution Percentage:")
print(df['is_fraud'].value_counts(normalize=True) * 100) # Corrected column name

# Separate features (X) and target (y)
X = df.drop('is_fraud', axis=1)
y = df['is_fraud'] # Corrected column name

# --- Preprocessing: Identify and drop non-numeric columns for scaling ---
# The StandardScaler expects numerical input. Columns like 'trans_date_trans_time', 'merchant', 'category', etc.
# are not directly suitable for scaling without further encoding or feature engineering.
# For now, we will drop them to allow the scaling to proceed.

# Identify non-numeric columns
non_numeric_cols = X.select_dtypes(include=['object', 'datetime64[ns]']).columns
print(f"\nDropping non-numeric columns before scaling: {list(non_numeric_cols)}")
X = X.drop(columns=non_numeric_cols)

# --- 2. Data Splitting ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"\nTraining set class distribution:\n{y_train.value_counts(normalize=True) * 100}")
print(f"Test set class distribution:\n{y_test.value_counts(normalize=True) * 100}")

# --- 3. Data Preprocessing (Scaling) ---
# Scale numerical features. This is crucial for Logistic Regression and can help other models.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 4. Handle Class Imbalance (SMOTE) ---
# Fraud detection datasets are typically highly imbalanced. SMOTE can help improve minority class recall.
print("\nApplying SMOTE to training data...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Original training set shape: {X_train_scaled.shape}, {y_train.shape}")
print(f"Resampled training set shape: {X_train_resampled.shape}, {y_train_resampled.shape}")
print(f"Resampled training set class distribution:\n{y_train_resampled.value_counts(normalize=True) * 100}")


# --- 5. Model Training and Evaluation ---

models = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear', class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced')
}

results = {}

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1] # Probability of the positive class (fraud)

    print(f"\n{name} Classification Report:")
    print(classification_report(y_test, y_pred))

    print(f"{name} Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    # Extracting TN, FP, FN, TP for easier understanding
    tn, fp, fn, tp = cm.ravel()
    print(f"  True Negatives (Legitimate Correctly Predicted): {tn}")
    print(f"  False Positives (Legitimate Predicted as Fraud): {fp}")
    print(f"  False Negatives (Fraud Predicted as Legitimate): {fn}")
    print(f"  True Positives (Fraud Correctly Predicted): {tp}")

    # ROC AUC Score
    roc_auc = roc_auc_score(y_test, y_proba)
    print(f"{name} ROC AUC Score: {roc_auc:.4f}")

    # Precision-Recall AUC (often more informative for highly imbalanced datasets)
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall, precision)
    print(f"{name} Precision-Recall AUC: {pr_auc:.4f}")

    results[name] = {
        'classification_report': classification_report(y_test, y_pred, output_dict=True),
        'confusion_matrix': cm.tolist(),
        'roc_auc': roc_auc,
        'pr_auc': pr_auc
    }

print("\n--- Summary of Model Performance ---")
print("Note: Focus on metrics for the minority class (class 1 - Fraud) and AUC scores due to imbalance.")
for name, metrics in results.items():
    print(f"\nModel: {name}")
    print(f"  ROC AUC: {metrics['roc_auc']:.4f}")
    print(f"  Precision-Recall AUC: {metrics['pr_auc']:.4f}")
    if '1' in metrics['classification_report']:
        print(f"  F1-score (Fraud class): {metrics['classification_report']['1']['f1-score']:.4f}")
        print(f"  Recall (Fraud class): {metrics['classification_report']['1']['recall']:.4f}")
        print(f"  Precision (Fraud class): {metrics['classification_report']['1']['precision']:.4f}")
    else:
        print("  Fraud class not predicted by this model.")


Using Colab cache for faster access to the 'fraud-detection' dataset.
Path to dataset files: /kaggle/input/fraud-detection
Contents of /kaggle/input/fraud-detection: ['fraudTest.csv', 'fraudTrain.csv']

Dataset Head:
   Unnamed: 0 trans_date_trans_time            cc_num  \
0           0   2019-01-01 00:00:18  2703186189652095   
1           1   2019-01-01 00:00:44      630423337322   
2           2   2019-01-01 00:00:51    38859492057661   
3           3   2019-01-01 00:01:16  3534093764340240   
4           4   2019-01-01 00:03:06   375534208663984   

                             merchant       category     amt      first  \
0          fraud_Rippin, Kub and Mann       misc_net    4.97   Jennifer   
1     fraud_Heller, Gutmann and Zieme    grocery_pos  107.23  Stephanie   
2                fraud_Lind-Buckridge  entertainment  220.11     Edward   
3  fraud_Kutch, Hermiston and Farrell  gas_transport   45.00     Jeremy   
4                 fraud_Keeling-Crist       misc_pos   41.96     